# Boltz docking

## Setup

In [ ]:
#| default_exp boltz.dock

In [ ]:
#| export
# basics
import subprocess
from pathlib import Path

## CLI

Obtain API key from [boltz.bio](https://lab.boltz.bio/)

```python
# Install the CLI
pip install boltz-lab

# Set your API key
boltz-lab config --api-key "boltzpk_live_A3L3caM_...TRUNCATED..."

# Submit without waiting, good for virtual screening
boltz-lab predict job.yaml --no-wait --name "my_custom_name"

# waits + downloads locally, for single run
boltz-lab predict job.yaml --output ./results --name "my_custom_name"

# List all jobs
boltz-lab list

# Check status later
boltz-lab status <prediction-id>

# Download when complete
boltz-lab download <prediction-id> --output ./results
```

## Prepare YAML

In [ ]:
#| export
def prepare_boltz(seq: str, # Amino acid sequence of the protein the protein
                    smiles: str, # SMILES string of the ligand
                    fname: str, # Output filename (should end with .yaml)
                    ):
    "Create a YAML file for protein-ligand affinity prediction."
    yaml_content = f"""version: 1
sequences:
  - protein:
      id: "A"
      sequence: "{seq}"
  - ligand:
      id: "B"
      smiles: "{smiles}"
properties:
  - affinity:
      binder: "B"
"""
    with open(fname, "w") as f:
        f.write(yaml_content)

```python
from tqdm import tqdm
tqdm.pandas()

# mutate G12D from human WT seq
seq = "MTEYKLVVVGADGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKCDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQRVEDAFYTLVREIRQYRLKKISKEEKTPGCVKIKKCIIM"
df.progress_apply(lambda r: prepare_boltz(seq,r.SMILES,f"kras_g12d/{r.ID}.yaml") ,axis=1)
```

## Run in batch

In [ ]:
#| export
def run_boltz(file_list:list[Path], # list of .yaml path in Pathlib object
                     api_key,# API key for Boltz-Lab
                     job_name=None, # job name appeared in boltz
                     ):
    
    "Run Boltz-Lab predictions for a list of YAML files."

    # config key
    subprocess.run(
        ["boltz-lab", "config", "--api-key", api_key.strip()],
        check=True
    )
    failed = []

    for file in file_list:
        print(f"\nSubmitting: {str(file)}")

        result = subprocess.run(
            ["boltz-lab", "predict", str(file),
             "--no-wait", # for batch run, so no need to wait the results til the next
             "--name",file.stem if job_name is None else job_name, # job name appeared in boltz
             ],
            capture_output=True,
            text=True,
        )

        if result.returncode != 0: failed.append(file.name)

        print(result.stdout)


    print("\n======== SUMMARY ========")
    print(f"Total: {len(file_list)}")
    print(f"Failed: {len(failed)}")

    return failed


```python
from fastcore.all import L

# suppose the yaml files are under a single folder
file_list = L(sorted(Path('kras').glob("*.yaml")))

run_boltz(file_list,key)
```

## Results analysis

Download results either from website sandbox, or through CLI

Get optimization score and affinity score, merge with df that contains experimental data

In [ ]:
#| export
import matplotlib.pyplot as plt
import seaborn as sns, numpy as np
from scipy.stats import spearmanr

In [ ]:
#| export
def plot_scatter_spearman(data, x, y, ax=None):
    """
    Plot scatter + Spearman correlation and p-value annotation.
    """
    if ax is None:
        ax = plt.gca()

    # Drop NA
    sub = data[[x, y]].dropna()
    x_vals = sub[x]
    y_vals = sub[y]

    # Compute Spearman
    rho, p = spearmanr(x_vals, y_vals)

    # Plot
    sns.scatterplot(data=sub, x=x, y=y, ax=ax)

    # Annotate
    text = f"Spearman ρ = {rho:.3f}\np = {p:.2e}"
    ax.text(
        0.98, 0.98,   # x, y in axes fraction
        text,
        transform=ax.transAxes,
        ha='right',          # horizontal align
        va='top',            # vertical align
        fontsize=11,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.7)
    )

    ax.set_title(f'{x} vs {y}')

    return rho, p

```python
x_vars = ['log10_Kd', 'log10_IC50', 'log10_erk_IC50']
y_vars = ['Optimization', 'Binding', 'Structure_confidence']

fig, axes = plt.subplots(len(x_vars), len(y_vars), figsize=(18, 18))

for i, x_var in enumerate(x_vars):
    for j, y_var in enumerate(y_vars):
        plot_scatter_spearman(df, x_var, y_var, ax=axes[i, j])

plt.tight_layout()
plt.show()
```

## End

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()